# BiLSTM Arabic Diacritization Pipeline (PyTorch)

This notebook implements a character-level BiLSTM pipeline for Arabic text diacritization using PyTorch.
Edit the `DATA_PATH` variables in the first code cell to point to your dataset.

* The only actual input feature to the model is:

        A 25-dimensional embedding vector representing each character.

* For each position in the sequence:

    1) Take the character ID (like "ل", "س", "و", <SPACE>, <NONAR>, etc.)

    2) Convert it to a 25-dimensional vector using the embedding layer.

    3) Feed these 25-dim vectors into the BiLSTM from left-to-right and right-to-left.

* So the network learns:

    1) context from characters before the position (forward LSTM)

    2) context from characters after the position (backward LSTM)

    Then it merges the two directions → a contextual representation of the character.

    Finally, the model outputs a diacritic class for that timestep.

* Arabic diacritization is context-dependent, e.g.

    1) “عَلِمَ” vs “عُلِمَ” (vowel depends on verb form)

    2) “في البيتِ” vs “في البيتُ” (case endings depend on grammar)

    3) with Shadda or without depending on morphological assimilation

* The BiLSTM captures these dependencies because:

    1) embeddings encode the character identity

    2) LSTM hidden states encode the sequence context


In [4]:
import os, glob, json
import unicodedata
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# DATA_PATH = '/kaggle/input/tashkeel-dataset/dataset/'
DATA_PATH = 'dataset/'
TRAIN_FILE = os.path.join(DATA_PATH, 'train.txt')
VAL_FILE = os.path.join(DATA_PATH, 'val.txt')
OUTPUT_MODEL_PATH = '/kaggle/working/bilstm_diac_pytorch_with_der.pt'

# Hyperparameters
MAXLEN = 500
EMBED_DIM = 25
LSTM_UNITS = 256
FF_UNITS = 512
DROPOUT = 0.5
BATCH_SIZE = 256
EPOCHS = 50
PLACEHOLDER = '<NONAR>'
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'
SPACE_TOKEN = '<SPACE>'


Using device: cpu


In [ ]:
# Check if a character is a combining diacritic
def is_combining(ch):
    return unicodedata.category(ch) == 'Mn'

# Split a string into (base_char, diacritics) pairs
def split_char_diacritic_pairs(sentence):
    pairs = []
    base = None
    diacs = ''
    for ch in sentence:
        if is_combining(ch):
            if base is None:
                base = '<UNK_BASE>'
            diacs += ch
        else:
            if base is not None:
                pairs.append((base, diacs))
            base = ch
            diacs = ''
    # Avoid losing the last pair
    if base is not None:
        pairs.append((base, diacs))
    return pairs

# Check if a character is an Arabic letter
def is_arabic_letter(ch):
    # Check if ch is a single character
    if not isinstance(ch, str) or len(ch) != 1:
        return False
    # Converts the character into its Unicode code point
    code = ord(ch)
    return (
        (0x0600 <= code <= 0x06FF) or
        (0x0750 <= code <= 0x077F) or
        (0x08A0 <= code <= 0x08FF) or
        (0xFB50 <= code <= 0xFDFF) or
        (0xFE70 <= code <= 0xFEFF)
    )

# Transform (base_char, diacritics) pairs with placeholders
def placeholder_transform_pairs(pairs, placeholder=PLACEHOLDER):
    tokens = []
    labels = []
    for base, d in pairs:
        # Handle <UNK_BASE>
        if isinstance(base, str) and base.startswith('<') and base.endswith('>'):
            tokens.append(base)
            labels.append('')
        # Handle space
        elif base.isspace():
            tokens.append(SPACE_TOKEN)
            labels.append('')
        # Handle Arabic letters and Tatweel
        elif is_arabic_letter(base) or base == 'ـ':
            tokens.append(base)
            labels.append(d)
        # Handle other non-Arabic characters as Numbers
        else:
            tokens.append(placeholder)
            labels.append('')
    return tokens, labels


In [6]:
# Load train and val data
def load_lines_from_file(path):
    lines = []
    with open(path, 'r', encoding='utf8') as fh:
        for line in fh:
            s = line.strip()
            if s:
                lines.append(s)
    return lines

train_lines = load_lines_from_file(TRAIN_FILE) if os.path.exists(TRAIN_FILE) else []
val_lines = load_lines_from_file(VAL_FILE) if os.path.exists(VAL_FILE) else []
print('Train lines:', len(train_lines), 'Val lines:', len(val_lines))

train_tokens, train_labels = [], []
for line in train_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    train_tokens.append(toks)
    train_labels.append(labs)

val_tokens, val_labels = [], []
for line in val_lines:
    pairs = split_char_diacritic_pairs(line)
    toks, labs = placeholder_transform_pairs(pairs)
    val_tokens.append(toks)
    val_labels.append(labs)

print('Example tokens (first train line):', train_tokens[0][:60] if len(train_tokens)>0 else '---')


Train lines: 50000 Val lines: 2500
Example tokens (first train line): ['و', 'ل', 'و', '<SPACE>', 'ج', 'م', 'ع', '<SPACE>', 'ث', 'م', '<SPACE>', 'ع', 'ل', 'م', '<SPACE>', 'ت', 'ر', 'ك', '<SPACE>', 'ر', 'ك', 'ن', '<SPACE>', 'م', 'ن', '<SPACE>', 'ا', 'ل', 'أ', 'و', 'ل', 'ى', '<SPACE>', 'ب', 'ط', 'ل', 'ت', 'ا', '<SPACE>', 'و', 'ي', 'ع', 'ي', 'د', 'ه', 'م', 'ا', '<SPACE>', 'ج', 'ا', 'م', 'ع', 'ا', '<SPACE>', '،', '<SPACE>', 'أ', 'و', '<SPACE>', 'م']


In [7]:
# Build vocabularies

# Calculate character frequencies
char_counter = Counter(token for seq in (train_tokens + val_tokens) for token in seq)

special_chars = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN, SPACE_TOKEN, PLACEHOLDER]
single_chars = sorted([c for c in char_counter if len(c) == 1 and c not in special_chars])
multi_chars = sorted([c for c in char_counter if len(c) != 1 and c not in special_chars])

# Create char2idx and idx2char mappings
chars = special_chars + single_chars + multi_chars
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = {i:c for c,i in char2idx.items()}

all_diacs = [d for lab in train_labels for d in lab]
diac_counter = Counter(all_diacs)
# <PAD_LABEL> -> For Padding the sequence and <NONE> -> No Diacritic
diac_classes = ['<PAD_LABEL>', '<NONE>']
most_common_diacs = [d for d,_ in diac_counter.most_common(13) if d != '']
for d in most_common_diacs:
    if d not in diac_classes:
        diac_classes.append(d)
diac_classes.append('<OTHER>')
diac2idx = {d:i for i,d in enumerate(diac_classes)}
idx2diac = {i:d for d,i in diac2idx.items()}

print('Vocab size:', len(char2idx), 'Diac labels:', len(diac2idx))


Vocab size: 45 Diac labels: 15


In [8]:
# Map diacritic to label index
def map_diacritic_to_label(d):
    if d == '':
        return diac2idx['<NONE>']
    if d in diac2idx:
        return diac2idx[d]
    return diac2idx['<OTHER>']

def tokens_labels_to_ids(sequences_tokens, sequences_labels, maxlen=MAXLEN):
    X = []
    y = []
    for toks, labs in zip(sequences_tokens, sequences_labels):
        # Add SOS and EOS tokens
        seq_mod = [SOS_TOKEN] + toks + [EOS_TOKEN]
        lab_mod = [''] + labs + ['']

        # Convert to IDs, handling unknown tokens
        x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
        y_ids = [map_diacritic_to_label(d) for d in lab_mod]

        # Truncate sequences if they exceed maxlen, to have consistent input size
        if len(x_ids) > maxlen:
            x_ids = x_ids[:maxlen]
            y_ids = y_ids[:maxlen]
        X.append(x_ids)
        y.append(y_ids)
    
    # Pad sequences to maxlen
    X_pad = []
    y_pad = []
    for seq_ids in X:
        if len(seq_ids) < maxlen:
            seq_ids = seq_ids + [char2idx[PAD_TOKEN]] * (maxlen - len(seq_ids))
        X_pad.append(seq_ids)
    for lab_ids in y:
        if len(lab_ids) < maxlen:
            lab_ids = lab_ids + [diac2idx['<PAD_LABEL>']] * (maxlen - len(lab_ids))
        y_pad.append(lab_ids)
    return np.array(X_pad, dtype=np.int64), np.array(y_pad, dtype=np.int64)

X_train_pad, y_train_pad = tokens_labels_to_ids(train_tokens, train_labels, MAXLEN)
X_val_pad, y_val_pad = tokens_labels_to_ids(val_tokens, val_labels, MAXLEN)
print('Shapes:', X_train_pad.shape, y_train_pad.shape, X_val_pad.shape, y_val_pad.shape)


Shapes: (50000, 500) (50000, 500) (2500, 500) (2500, 500)


In [ ]:
# A PyTorch Dataset wrapper for DataLoader to handle character and diacritic sequences
class CharDiacDataset(Dataset):
    def __init__(self, X_arr, y_arr):
        self.X = X_arr
        self.y = y_arr
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

# Collate function to pad sequences in a batch
def collate_pad(batch, pad_idx=0, pad_label_idx=0):
    Xs, Ys = zip(*batch)
    # Store original lengths before padding for LSTM packing
    lengths = [x.size(0) for x in Xs]
    maxlen = max(lengths)
    Xp = torch.full((len(Xs), maxlen), pad_idx, dtype=torch.long)
    Yp = torch.full((len(Xs), maxlen), pad_label_idx, dtype=torch.long)
    for i,(x,y) in enumerate(zip(Xs,Ys)):
        Xp[i,:x.size(0)] = x
        Yp[i,:y.size(0)] = y
    return Xp, Yp, torch.tensor(lengths, dtype=torch.long)

train_dataset = CharDiacDataset(X_train_pad, y_train_pad)
val_dataset = CharDiacDataset(X_val_pad, y_val_pad)
train_loader = DataLoader(train_dataset, batch_size=min(64, len(train_dataset)), shuffle=True, collate_fn=lambda b: collate_pad(b, char2idx[PAD_TOKEN], diac2idx['<PAD_LABEL>']))
val_loader = DataLoader(val_dataset, batch_size=min(64, len(val_dataset)), shuffle=False, collate_fn=lambda b: collate_pad(b, char2idx[PAD_TOKEN], diac2idx['<PAD_LABEL>']))
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))


In [ ]:
class BiLSTM_Diac(nn.Module):
    def __init__(self, vocab_size, emb_dim, lstm_units, ff_units, num_labels, pad_idx=0, dropout=0.5):
        super().__init__()
        # Map input character indices to dense embeddings, ensuring padding_idx is not updated during training
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        # BiDirectional LSTM layers, with dropout for regularization (Avoid overfitting & memorization by randomly dropping units)
        self.bilstm1 = nn.LSTM(emb_dim, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout)
        self.bilstm2 = nn.LSTM(2*lstm_units, lstm_units, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)

        # Feedforward layers to mix LSTM outputs to diacritic label logits
        self.ff1 = nn.Linear(2*lstm_units, ff_units)
        self.ff2 = nn.Linear(ff_units, ff_units)
        self.out = nn.Linear(ff_units, num_labels)
        self.relu = nn.ReLU()
    # Forward pass
    def forward(self, x, lengths=None):
        emb = self.embedding(x)
        if lengths is not None:
            # Pack padded sequence for efficient processing by LSTM
            packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out1, _ = self.bilstm1(packed)
            # Unpack the sequence back to padded form
            out1, _ = nn.utils.rnn.pad_packed_sequence(packed_out1, batch_first=True)
        else:
            # Directly pass embeddings through the first BiLSTM layer, Used in validation when all sequences are of same length
            out1, _ = self.bilstm1(emb)
        out1 = self.dropout1(out1)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(out1, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out2, _ = self.bilstm2(packed)
            out2, _ = nn.utils.rnn.pad_packed_sequence(packed_out2, batch_first=True)
        else:
            out2, _ = self.bilstm2(out1)
        out2 = self.dropout2(out2)
        # Feedforward layers with ReLU activations
        ff = self.relu(self.ff1(out2))
        ff = self.relu(self.ff2(ff))
        logits = self.out(ff)
        return logits

vocab_size = len(char2idx)
num_labels = len(diac2idx)
pad_idx = char2idx[PAD_TOKEN]
pad_label_idx = diac2idx['<PAD_LABEL>']
model = BiLSTM_Diac(vocab_size=vocab_size, emb_dim=EMBED_DIM, lstm_units=LSTM_UNITS, ff_units=FF_UNITS, num_labels=num_labels, pad_idx=pad_idx, dropout=DROPOUT)
model.to(device)
print(model)


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, grad_clip=None):
    model.train()
    total_loss = 0.0
    count = 0
    for X, Y, lengths in dataloader:
        X = X.to(device); Y = Y.to(device); lengths = lengths.to(device)
        # Clear gradients
        optimizer.zero_grad()
        # Forward pass
        logits = model(X, lengths)
        b, T, C = logits.shape
        logits_flat = logits.view(-1, C)
        labels_flat = Y.view(-1)
        # Compute loss and backpropagate
        loss = criterion(logits_flat, labels_flat)
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        # Update model parameters using Adam optimizer
        optimizer.step()
        total_loss += loss.item()
        count += 1
    return total_loss / max(1, count)

# Evaluate model on validation set, No gradient computation
def eval_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    count = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, Y, lengths in dataloader:
            X = X.to(device); Y = Y.to(device); lengths = lengths.to(device)
            logits = model(X, lengths)
            b, T, C = logits.shape
            logits_flat = logits.view(-1, C)
            labels_flat = Y.view(-1)
            loss = criterion(logits_flat, labels_flat)
            total_loss += loss.item()
            count += 1
            preds = logits.argmax(dim=-1)
            mask = (labels_flat != pad_label_idx)
            correct += (preds.view(-1)[mask] == labels_flat[mask]).sum().item()
            total += mask.sum().item()
    acc = correct / total if total>0 else 0.0
    return total_loss / max(1,count), acc


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=pad_label_idx)
best_val_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_epoch(model, train_loader, optimizer, criterion, device, grad_clip=5.0)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    print(f'Epoch {epoch} train_loss={tr_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'model_state_dict': model.state_dict(), 'char2idx': char2idx, 'diac2idx': diac2idx}, OUTPUT_MODEL_PATH)
        print('Saved best model to', OUTPUT_MODEL_PATH)


In [ ]:
# DER Evaluation

def predict_on_loader_collect(model, loader, device):
    model.eval()
    Xs, Ys, Ypreds = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, lengths in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch, lengths.to(device))
            pred_ids = logits.argmax(dim=-1).cpu().numpy()
            Xs.append(X_batch.cpu().numpy())
            Ys.append(y_batch.numpy())
            Ypreds.append(pred_ids)
    if len(Xs) == 0:
        return None, None, None
    X_all = np.vstack(Xs)
    Y_true_all = np.vstack(Ys)
    Y_pred_all = np.vstack(Ypreds)
    return X_all, Y_true_all, Y_pred_all

def compute_der_from_arrays(X_pad, y_true_pad, y_pred_pad, char2idx, diac2idx, idx2diac, exclude_case_ending=False):
    pad_label_id = diac2idx["<PAD_LABEL>"]
    pad_char_id = char2idx[PAD_TOKEN]
    sos_id = char2idx.get(SOS_TOKEN, None)
    eos_id = char2idx.get(EOS_TOKEN, None)
    space_id = char2idx.get(SPACE_TOKEN, None)
    placeholder_id = char2idx.get(PLACEHOLDER, None)

    N, T = X_pad.shape
    total = 0
    incorrect = 0

    for i in range(N):
        for t in range(T):
            ch_id = int(X_pad[i, t])
            if ch_id == pad_char_id:
                break
            if sos_id is not None and ch_id == sos_id:
                continue
            if eos_id is not None and ch_id == eos_id:
                continue

            gold = int(y_true_pad[i, t])
            if gold == pad_label_id:
                continue

            if exclude_case_ending:
                next_id = pad_char_id
                if t + 1 < T:
                    next_id = int(X_pad[i, t+1])
                is_word_final = False
                if next_id == pad_char_id:
                    is_word_final = True
                elif space_id is not None and next_id == space_id:
                    is_word_final = True
                elif placeholder_id is not None and next_id == placeholder_id:
                    is_word_final = True
                elif eos_id is not None and next_id == eos_id:
                    is_word_final = True

                if is_word_final:
                    continue

            pred = int(y_pred_pad[i, t])
            total += 1
            if pred != gold:
                incorrect += 1

    der = (incorrect / total) * 100.0 if total > 0 else None
    return der, incorrect, total

def evaluate_with_der(model, loader, device, name="validation"):
    try:
        val_loss, val_acc = eval_epoch(model, loader, criterion, device)
    except Exception:
        val_loss, val_acc = None, None

    X_all, y_true_all, y_pred_all = predict_on_loader_collect(model, loader, device)
    if X_all is None:
        print(f"No data found in {name} loader.")
        return

    none_id = diac2idx.get("<NONE>", None)
    if none_id is not None:
        gold_non_none = (y_true_all != none_id).sum()
        if gold_non_none == 0:
            print(f"Dataset in {name} loader appears to have NO gold diacritics (all labels are <NONE>).")
            print("DER cannot be computed for this loader. You can still compute accuracy if desired.")
            if val_acc is not None:
                print(f"Label-level accuracy (existing): {val_acc:.4f}")
            return

    der_all, incorrect_all, total_all = compute_der_from_arrays(
        X_all, y_true_all, y_pred_all, char2idx, diac2idx, idx2diac, exclude_case_ending=False
    )
    der_no_ce, incorrect_no_ce, total_no_ce = compute_der_from_arrays(
        X_all, y_true_all, y_pred_all, char2idx, diac2idx, idx2diac, exclude_case_ending=True
    )

    header = f"Evaluation on {name} set"
    print("="*len(header))
    print(header)
    print("="*len(header))
    if val_loss is not None:
        print(f"Label-level loss: {val_loss:.4f}")
    if val_acc is not None:
        print(f"Label-level accuracy (existing metric): {val_acc:.4f}")
    print(f"DER (all positions): {der_all:.4f}%   ({incorrect_all} incorrect / {total_all} total positions)")
    print(f"DER (no case ending): {der_no_ce:.4f}%   ({incorrect_no_ce} incorrect / {total_no_ce} total positions)")
    print()

# Example usage (after training): evaluate on validation and test
try:
    evaluate_with_der(model, val_loader, device, name='validation')
except NameError:
    print('val_loader or model not found in this notebook context; run training cells first, then run this cell.')


In [17]:
# =======================
# Inference on raw text using trained model
# =======================
import torch
import unicodedata

# --------- 1. Load checkpoint (weights + vocab) ----------
MODEL_PATH = "bilstm_diac_pytorch_with_der.pt"  # <-- change to your actual path

ckpt = torch.load(MODEL_PATH, map_location=device)

# checkpoint was saved like {"model_state_dict": ..., "char2idx": ..., "diac2idx": ...}
char2idx = ckpt["char2idx"]
diac2idx = ckpt["diac2idx"]
idx2diac = {v: k for k, v in diac2idx.items()}  # inverse mapping: id -> label string

print(f"✅ checkpoint loaded | chars={len(char2idx)}, diacs={len(diac2idx)}")

# --------- 2. Rebuild the model with correct sizes ----------
vocab_size = len(char2idx)
num_labels = len(diac2idx)
pad_idx = char2idx[PAD_TOKEN]


# NOTE: use the same class and hyperparams as training
model = BiLSTM_Diac(
    vocab_size=vocab_size,
    emb_dim=EMBED_DIM,
    lstm_units=LSTM_UNITS,
    ff_units=FF_UNITS,
    num_labels=num_labels,
    pad_idx=pad_idx,
    dropout=DROPOUT,
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("✅ trained model weights loaded")

# --------- 3. Helpers for inference on raw text ----------
def prepare_inference_raw_line(raw_line, placeholder=PLACEHOLDER):
    tokens = []
    original_nonar = []
    for ch in raw_line:
        if is_arabic_letter(ch) or ch == "ـ":
            tokens.append(ch)
        elif ch.isspace():
            tokens.append(SPACE_TOKEN)
            original_nonar.append((len(tokens) - 1, ch))
        else:
            tokens.append(placeholder)
            original_nonar.append((len(tokens) - 1, ch))
    return tokens, original_nonar


# if your diacritic labels are already combining marks, this is enough
def label_to_mark(lab):
    # treat '<NONE>' or empty as no diacritic
    if not lab or lab == "<NONE>":
        return ""
    # if all chars are combining marks, keep them
    if all(unicodedata.category(ch) in ("Mn", "Mc") for ch in lab):
        return lab
    # otherwise just return lab (or map by name if you used names like 'FATHA')
    return lab

def infer_and_reconstruct(raw_line):
    # prepare tokens + placeholder positions
    tokens, original_nonar = prepare_inference_raw_line(raw_line)
    seq_mod = [SOS_TOKEN] + tokens + [EOS_TOKEN]

    x_ids = [char2idx.get(t, char2idx[UNK_TOKEN]) for t in seq_mod]
    if len(x_ids) < MAXLEN:
        x_ids = x_ids + [char2idx[PAD_TOKEN]] * (MAXLEN - len(x_ids))
    else:
        x_ids = x_ids[:MAXLEN]

    x_tensor = torch.tensor([x_ids], dtype=torch.long).to(device)
    lengths = torch.tensor([min(len(seq_mod), MAXLEN)], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(x_tensor, lengths)
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    pred_diacs = [idx2diac.get(int(i), "<NONE>") for i in pred_ids]
    # skip SOS and align with tokens
    pred_trim = pred_diacs[1 : 1 + len(tokens)]

    # build diacritized string token-by-token
    out_tokens = []
    for i, tok in enumerate(tokens):
        if tok == SPACE_TOKEN:
            out_tokens.append(" ")
            continue
        if tok == PLACEHOLDER:
            # restore original non-Arabic character if available
            repl = next((ch for pos, ch in original_nonar if pos == i), PLACEHOLDER)
            out_tokens.append(repl)
            continue
        # normal Arabic char
        lab = pred_trim[i] if i < len(pred_trim) else "<NONE>"
        mark = label_to_mark(lab)
        combined = unicodedata.normalize("NFC", tok + mark)
        out_tokens.append(combined)

    return "".join(out_tokens)

# --------- 4. Test on your custom raw text (no training required) ----------
# take first 4 line from file 'dataset/test_no_diacritics.txt' if exists
test_file_path = os.path.join("", 'test_no_diacritics.txt')
raw_text = ""
if os.path.exists(test_file_path):
    with open(test_file_path, 'r', encoding='utf8') as f:
        for _ in range(20):
            line = f.readline()
            if not line:
                break
            raw_text += line

for line in raw_text.splitlines():
    line = line.strip()
    if not line:
        print()
        continue
    print("RAW:  ", line)
    print("DIAC: ", infer_and_reconstruct(line))
    print()


C:\Users\menna\AppData\Local\Temp\ipykernel_89180\1665076258.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(MODEL_PATH, map_location=device)


✅ checkpoint loaded | chars=45, diacs=15
✅ trained model weights loaded
RAW:   ليس للوكيل بالقبض أن يبرأ المدين أو يهب الدين له أو يأخذ رهنا من المدين في مقابل الدين أو يقبل إحالته على شخص آخر لكن له أن يأخذ كفيلا لكن ليس له أن يأخذ كفيلا بشرط براءة الأصيل انظر المادة ( 648 ) ( الأنقروي ، الطحطاوي وصرة الفتاوى ، البحر ) .
DIAC:  لَيْسَ لِلْوَكِيلِ بِالْقَبْضِ أَنْ يَبْرَأَ الْمَدِينَ أَوْ يَهَبَ الدَّيْنَ لَهُ أَوْ يَأْخُذَ رَهْنًا مِنْ الْمَدِينِ فِي مُقَابِلِ الدَّيْنِ أَوْ يَقْبَلَ إحَالَتُهُ عَلَى شَخْصٍ آخَرَ لَكِنْ لَهُ أَنْ يَأْخُذَ كَفِيلًا لَكِنْ لَيْسَ لَهُ أَنْ يَأْخُذَ كَفِيلًا بِشَرْطِ بَرَاءَةِ الْأَصِيلِ اُنْظُرْ الْمَادَّةَ ( 648 ) ( الْأَنْقِرْوِيُّ ، الطَّحْطَاوِيُّ وَصُرَّةُ الْفَتَاوَى ، الْبَحْرُ ) .

RAW:   ( قوله ويقع في بعض النسخ بمنفعة ومعين ) أي : أوصى بمجموع شيئين بمنفعة شيء وبمعين وقوله وليس ذلك بصحيح كأن عدم الصحة من جهة أن هذه المسألة فيها نص بهذا الحكم الذي أشار إليه المصنف بقوله وإن أوصى بمنفعة معين وبعض شيوخنا علل عدم الصحة بقوله لما علمت من اختلاف الحك